# Evaluating A/B Test

After we have collected enough samples for the A/B test, here we will evaluate the result using statistical testing.

In [1]:
import numpy as np
import pandas as pd
import os

pd.options.display.max_columns = 999
pd.options.display.float_format = "{:.2f}".format

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

# where is your data
data_path = '/content/drive/My Drive/databank/'

Mounted at /content/drive/


The following is the data for the News A/B Testing to increase the subscripition rate

In [3]:
df = pd.read_csv(data_path + 'news_ab_test.csv')

df.head()

,user_id,group,device,gender,time_spent,session_duration,page_views,conversion
0,C79294,Variant,Desktop,Male,52.62,12.83,20,0
1,C36578,Variant,Desktop,Male,54.88,12.00,19,0
2,C80909,Control,Mobile,Male,70.51,7.76,12,0
3,C86020,Control,Mobile,Male,56.89,6.37,12,0
4,C48900,Control,Mobile,Male,46.07,9.24,10,0


Data description:

* user_id: ID of the user
* group: the group assigned to each user (control/variant)
* device: device used by the user (desktop/mobile)
* time_spent: total time spent on the app during experiment
* page_views: total page viewed by user during experiment
* conversion: whether the user subscribe to the news (1) or not (0)

## Conversion Rate

First, we will evaluate if the conversion has increased with the new variant. To compare the proportion between two different samples, we can use the proportion z-test.

### Proportion Z Test


Since the test is evaluating whether the variant is better than the control (superiority test), we need to adjust our hypothesis statement into the following:

$H_0: p_{variant} \leq p_{control}$

$H_1: p_{variant} > p_{control}$


In [4]:
from statsmodels.stats.proportion import proportions_ztest

In [6]:
group_a = df[ df['group'] == 'Control']
group_b = df[ df['group'] == 'Variant']

conversion = [ group_b['conversion'].sum(), group_a['conversion'].sum() ]
sample = [ group_b['conversion'].shape[0], group_a['conversion'].shape[0] ]

# Perform the two-proportion Z-test
z_stat, p_value = proportions_ztest(conversion, sample, alternative='larger')

# Print results
print(f"P-value: {p_value:.4f}")
print(f"CVR control: {(group_a['conversion'].sum()/group_a['conversion'].shape[0]):.3f}")
print(f"CVR variant: {(group_b['conversion'].sum()/group_b['conversion'].shape[0]):.3f}")


P-value: 0.0000
CVR control: 0.100
CVR variant: 0.153



When the $H_0$ is rejected (p-value < 0.05), we can safely assume that the conversion rate of the variant is higher than the control group. We can also see the CVR variant is 15% while CVR for control group is 10%.


More example on two sample proportion z-test with manual calculation: [Youtube](https://youtu.be/pCbNUnZ98oE?si=aX92dVgMEqGwfCph)

## Session Duration

Now let's evaluate whether user in variant group has longer sesion duration compared to the control group

In [ ]:
session_a = df[ df['group'] == 'Control']['session_duration']
session_b = df[ df['group'] == 'Variant']['session_duration']

### Normality Test

The hypothesis testing for normality test:

$H_0$ : data is normally distributed

$H_1$ : data is not normally distributed

In [ ]:
from scipy import stats

In [ ]:
def ad_normal_test(x):
  # Perform the Anderson-Darling test
  result = stats.anderson(x, dist='norm')

  # Interpretation
  for cv, sig in zip(result.critical_values[2:], result.significance_level[2:]):
    if result.statistic < cv:
        print(f"Fail to reject normality at {sig}% significance level")
    else:
        print(f"Reject normality at {sig}% significance level")

In [ ]:
# session duration group control
ad_normal_test(session_a)

Fail to reject normality at 5.0% significance level
Fail to reject normality at 2.5% significance level
Fail to reject normality at 1.0% significance level


In [ ]:
# session duration group variant
ad_normal_test(session_b)

Fail to reject normality at 5.0% significance level
Fail to reject normality at 2.5% significance level
Fail to reject normality at 1.0% significance level


### Independent t-test

$
H_0 : \bar{x}_B \leq \bar{x}_A
$

$
H_1 : \bar{x}_B > \bar{x}_A
$


In [ ]:
# Perform independent t-test
t_stat, p_value = stats.ttest_ind(session_b, session_a, alternative = 'greater')

print(f"P-value: {p_value:.4f}")
print(f"Mean of session duration A: {session_a.mean():.2f}")
print(f"Mean of session duration B: {session_b.mean():.2f}")

P-value: 0.5987
Mean of session duration A: 10.00
Mean of session duration B: 10.00


## Page Views

In [ ]:
page_a = df[ df['group'] == 'Control']['page_views']
page_b = df[ df['group'] == 'Variant']['page_views']

In [ ]:
# page views group control
ad_normal_test(page_a)

Reject normality at 5.0% significance level
Reject normality at 2.5% significance level
Reject normality at 1.0% significance level


In [ ]:
# page views group control
ad_normal_test(page_b)

Reject normality at 5.0% significance level
Reject normality at 2.5% significance level
Reject normality at 1.0% significance level


### Mann-Whitney Test

In [ ]:
# Perform the Mann-Whitney U test
stat, p_value = stats.mannwhitneyu(page_b, page_a, alternative='greater')

# Print results
print(f"P-value: {p_value}")
print(f"Mean of page views A: {page_a.mean():.2f}")
print(f"Mean of page views B: {page_b.mean():.2f}")

P-value: 0.0
Mean of page views A: 9.49
Mean of page views B: 14.50


## Total Time Spent on Site

In [ ]:
time_a = df[ df['group'] == 'Control']['time_spent']
time_b = df[ df['group'] == 'Variant']['time_spent']

In [ ]:
ad_normal_test(time_a)

Reject normality at 5.0% significance level
Reject normality at 2.5% significance level
Reject normality at 1.0% significance level


In [ ]:
ad_normal_test(time_b)

Reject normality at 5.0% significance level
Reject normality at 2.5% significance level
Reject normality at 1.0% significance level


In [ ]:
# Perform the Mann-Whitney U test
stat, p_value = stats.mannwhitneyu(time_b, time_a, alternative='greater')

# Print results
print(f"P-value: {p_value}")
print(f"Mean of page views A: {time_a.mean():.2f}")
print(f"Mean of page views B: {time_b.mean():.2f}")

P-value: 0.0
Mean of page views A: 39.99
Mean of page views B: 65.06
